In [13]:
import pickle
from torchvision import datasets, transforms
from torch.utils.data import Subset, DataLoader


transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
full_dataset = datasets.ImageFolder("archive_extracted/split/train", transform=transform)


with open("dataset_splits.pkl","rb") as f:
    splits = pickle.load(f)

train_dataset = Subset(full_dataset, splits["train_idx"])
val_dataset   = Subset(full_dataset, splits["val_idx"])
test_dataset  = Subset(full_dataset, splits["test_idx"])
classes = splits["classes"]

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=False)
val_loader   = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=64, shuffle=False)


In [14]:


from torch.utils.data import DataLoader

BATCH_SIZE = 64
NUM_WORKERS = 4

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)


classes = getattr(full_dataset, "classes", None)
print("Classes:", classes)
print("Sizes:", len(train_dataset), len(val_dataset), len(test_dataset))


Classes: ['Mild', 'Moderate', 'No_DR', 'Proliferate_DR', 'Severe']
Sizes: 2343 292 294


In [15]:
import torch
import torch.nn as nn
from torchvision import models

IMG_SIZE = 224  

class ImageEncoder(nn.Module):
    def __init__(self, out_dim=256, pretrained=True, freeze_backbone=True):
        super().__init__()
        m = models.efficientnet_b0(
            weights=models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
        )
        feat_dim = 1280  
        self.backbone = nn.Sequential(
            m.features,   
            m.avgpool,    
            nn.Flatten(), 
        )
        self.proj = nn.Linear(feat_dim, out_dim)
        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False

    
    def forward(self, x, return_normed=True):
        feats = self.backbone(x)        # [B,1280]
        z = self.proj(feats)            # [B,256]
        if return_normed:
            z = nn.functional.normalize(z, dim=1)
        return z

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
encoder = ImageEncoder(out_dim=256, pretrained=True, freeze_backbone=True).to(device)
encoder.eval()
print("Encoder ready on", device)


Encoder ready on cpu


In [16]:
import os, time
import torch
from torch.utils.data import Subset

OUT_DIR = "./embeddings_out"
os.makedirs(OUT_DIR, exist_ok=True)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def subset_paths_and_labels(subset: Subset):
    
    base = subset.dataset  # full_dataset: ImageFolder
    idxs = subset.indices
    paths = [base.samples[i][0] for i in idxs]
    labels = torch.tensor([base.samples[i][1] for i in idxs], dtype=torch.long)
    classes = getattr(base, "classes", None)
    return paths, labels, classes

@torch.no_grad()
def extract_and_save_from_subset(split_name, subset, loader, encoder, out_dir):
    encoder.eval()
    all_embs = []

    t0 = time.time()
    for x, _ in loader:
        x = x.to(device, non_blocking=True)
        z = encoder(x, return_normed=True).cpu()   # [B,256]
        all_embs.append(z)

    embs = torch.cat(all_embs, dim=0)              # [N,256]
   
    paths, labels_ref, classes = subset_paths_and_labels(subset)

    
    assert embs.shape[0] == len(labels_ref), f"Length mismatch: {embs.shape[0]} vs {len(labels_ref)}"

    save_obj = {
        "embeddings": embs,              
        "labels": labels_ref,             
        "paths": paths,                   
        "classes": classes,               
        "encoder": "efficientnet_b0",
        "embed_dim": embs.shape[1],
        "img_size": IMG_SIZE,
        "normalized": True,
        "preprocess": {"mean": IMAGENET_MEAN, "std": IMAGENET_STD},
    }
    out_path = os.path.join(out_dir, f"{split_name}_embeddings.pt")
    torch.save(save_obj, out_path)
    print(f"[{split_name}] saved {embs.shape} -> {out_path}  ({time.time()-t0:.1f}s)")

extract_and_save_from_subset("train", train_dataset, train_loader, encoder, OUT_DIR)
extract_and_save_from_subset("val",   val_dataset,   val_loader,   encoder, OUT_DIR)
extract_and_save_from_subset("test",  test_dataset,  test_loader,  encoder, OUT_DIR)


[train] saved torch.Size([2343, 256]) -> ./embeddings_out\train_embeddings.pt  (55.7s)
[val] saved torch.Size([292, 256]) -> ./embeddings_out\val_embeddings.pt  (18.5s)
[test] saved torch.Size([294, 256]) -> ./embeddings_out\test_embeddings.pt  (17.7s)


Run the encoder separately on the train/val/test sets, saving each image's 256-dimensional vector, label, original path, and class name to a .pt file for use in fusion/visualization/federated tasks.

In [17]:
imgs, y = next(iter(train_loader))
with torch.no_grad():
    z = encoder(imgs.to(device))      
print("Input:", imgs.shape)           
print("Embedding:", z.shape)          
print("First vec (head):", z[0][:8])


Input: torch.Size([64, 3, 224, 224])
Embedding: torch.Size([64, 256])
First vec (head): tensor([ 0.0089,  0.0529, -0.0578,  0.0060,  0.1114, -0.0286, -0.0858, -0.0713])


In [18]:
encoder.train()
imgs, _ = next(iter(train_loader))
z = encoder(imgs.to(device))         
loss = z.pow(2).mean()
loss.backward()
print("proj layer has grad? ", encoder.proj.weight.grad is not None)  
encoder.eval()


proj layer has grad?  True


ImageEncoder(
  (backbone): Sequential(
    (0): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): SiLU(inplace=True)
      )
      (1): Sequential(
        (0): MBConv(
          (block): Sequential(
            (0): Conv2dNormActivation(
              (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
              (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
              (2): SiLU(inplace=True)
            )
            (1): SqueezeExcitation(
              (avgpool): AdaptiveAvgPool2d(output_size=1)
              (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
              (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
              (activation): SiLU(inplace=True)
              (scale_activati

In [ ]:
extract_and_save_from_subset("val", val_dataset, val_loader, encoder, "./embeddings_out")

import torch
ckpt = torch.load("./embeddings_out/val_embeddings.pt", map_location="cpu")
print(ckpt["embeddings"].shape, len(ckpt["labels"]), len(ckpt["paths"]), ckpt["classes"])



[val] saved torch.Size([292, 256]) -> ./embeddings_out\val_embeddings.pt  (17.5s)
torch.Size([292, 256]) 292 292 ['Mild', 'Moderate', 'No_DR', 'Proliferate_DR', 'Severe']
